# Mini-projet : détecter la redondance

| Module | Durée | Prérequis |
|---|---|---|
| M5 – Espaces vectoriels et bases | 150 min | M4 |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

**NutriScan** est une jeune entreprise qui développe une application de conseil alimentaire. Les fabricants lui envoient les valeurs nutritionnelles de leurs produits, pour 100 g, telles qu'elles figurent sur les emballages. Le fichier `donnees/produits.csv` en contient un premier échantillon : 42 produits et 10 colonnes numériques (énergie en kJ et en kcal, lipides, acides gras saturés, glucides, sucres, fibres, protéines, sel, sodium).

L'équipe données vous confie deux missions :

1. **Simplifier** : certaines colonnes se calculent-elles à partir des autres ? Si oui, l'application peut les recalculer au lieu de les stocker, et le modèle de recommandation n'en a pas besoin.
2. **Contrôler** : si des relations existent, elles permettent de repérer les fiches mal saisies.

On ne vous donne **aucune formule** : à vous de les retrouver dans les données.

**Activité** : Précisez les questions. Qu'appelle-t-on ici une colonne « redondante » ? Une relation vraie pour 40 produits sur 42 en est-elle une ? Quelles hypothèses faites-vous sur la précision des étiquettes ?

Indiquez vos réponses et justifiez-les.

## ② Estimer avant de calculer

In [ ]:
pari_nombre_relations = 2          # nombre de colonnes qui se calculent à partir des autres (modifiez)
pari_fiches_erronees = 0           # nombre de produits mal saisis (modifiez)

## ③ Abstraire

Chaque colonne est un vecteur de $\mathbb{R}^{42}$. Les questions de l'équipe se traduisent ainsi :

| Question de l'équipe | Traduction mathématique |
|---|---|
| Combien d'informations différentes ? | le **rang** de la matrice des données |
| Quelles colonnes se recalculent ? | les relations $A\mathbf{c} \approx \mathbf{0}$, c'est-à-dire le **noyau** (approché) |
| Quelles fiches sont fausses ? | les produits pour lesquels une relation, vraie partout ailleurs, est violée |

Deux difficultés, déjà rencontrées au notebook 5.2 :

- les étiquettes sont **arrondies** (l'énergie à l'unité, les nutriments au dixième de gramme) : les relations ne seront qu'approchées, il faut une tolérance ;
- les colonnes ont des **unités** très différentes (des kJ par centaines, du sodium en dixièmes de gramme). On les **standardise** d'abord : chaque colonne est centrée, puis divisée par son écart type. Centrer revient à ajouter une colonne constante : on détecte ainsi aussi les relations affines.

### Un outil en boîte noire : les valeurs singulières

La fonction `np.linalg.svd` calcule, pour une matrice, une liste de nombres positifs rangés par ordre décroissant, les **valeurs singulières**. Vous en verrez le fonctionnement au module M8 ; pour ce projet, une seule propriété suffit :

> Le nombre de valeurs singulières **proches de zéro** est le nombre de relations linéaires (presque) exactes entre les colonnes.

Une valeur singulière nulle signale une relation exacte ; une valeur très petite, une relation vraie aux arrondis près.

**Activité** : Pour la matrice de colonnes $\mathbf{u} = (1\,;\,0\,;\,2)$, $\mathbf{v} = (0\,;\,1\,;\,1)$ et $\mathbf{w} = \mathbf{u} + \mathbf{v}$, combien de valeurs singulières nulles attendez-vous ? Vérifiez dans la cellule suivante.

Indiquez vos réponses et justifiez-les.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

u, v = np.array([1., 0, 2]), np.array([0., 1, 1])
print("Valeurs singulières :", np.linalg.svd(np.column_stack([u, v, u + v]), compute_uv=False).round(6))

## ④ Calculer

### Découvrir les données

In [ ]:
produits = pd.read_csv("donnees/produits.csv")
colonnes = ["energie_kj", "energie_kcal", "lipides", "acides_gras_satures", "glucides",
            "sucres", "fibres", "proteines", "sel", "sodium"]
print(produits.shape)
produits.head()

In [ ]:
produits[colonnes].describe().round(2)

### Combien d'informations indépendantes ?

In [ ]:
def standardiser(tableau):
    return (tableau - tableau.mean()) / tableau.std()

def valeurs_singulieres(tableau):
    return np.linalg.svd(standardiser(tableau).to_numpy(), compute_uv=False)

sv = valeurs_singulieres(produits[colonnes])
print("Valeurs singulières :", sv.round(3))

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.semilogy(range(1, len(sv) + 1), sv, "o-")
ax.set_xlabel("rang de la valeur singulière"); ax.set_ylabel("valeur (échelle log)")
ax.set_title("Valeurs singulières des 10 colonnes standardisées"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

**Activité** : Combien de valeurs singulières sont nettement plus petites que les autres ? Qu'en concluez-vous, provisoirement, sur le rang ?

Indiquez vos réponses et justifiez-les.

### Quelles colonnes s'expliquent par les autres ?

Pour chaque colonne, on cherche la meilleure combinaison affine des neuf autres (`np.linalg.lstsq`, comme au notebook 5.2), et on mesure l'écart restant, rapporté à l'écart type de la colonne. Un écart proche de 0 signale une colonne redondante.

In [ ]:
def ajuster(tableau, cible, explicatives):
    # Meilleure combinaison affine des colonnes explicatives pour approcher la cible
    B = np.column_stack([np.ones(len(tableau)), tableau[explicatives].to_numpy(dtype=float)])
    y = tableau[cible].to_numpy(dtype=float)
    coef, *_ = np.linalg.lstsq(B, y, rcond=None)
    residus = y - B @ coef
    return pd.Series(coef, index=["constante"] + list(explicatives)), residus

ecarts = {}
for c in colonnes:
    autres = [k for k in colonnes if k != c]
    _, r = ajuster(produits, c, autres)
    ecarts[c] = np.std(r) / produits[c].std()
ecarts = pd.Series(ecarts).sort_values()

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.barh(ecarts.index, ecarts.values, color=[LIOUAI["vert"] if e < 0.05 else LIOUAI["gris"] for e in ecarts])
ax.set_xscale("log"); ax.set_xlabel("écart relatif restant (échelle log)")
ax.set_title("Chaque colonne, expliquée par les neuf autres")
plt.tight_layout(); plt.show()
ecarts.round(4)

**Activité** : Quelles colonnes semblent liées entre elles ? Formulez une hypothèse de relation pour chaque groupe, en vous appuyant sur ce que vous savez de l'énergie. Le sel et le sodium, ou l'énergie en kcal, devraient eux aussi être liés à d'autres colonnes, et ne le sont qu'en partie : quelle explication proposez-vous ?

Indiquez vos réponses et justifiez-les.

### Tester une relation, et repérer les fiches qui la violent

Testons l'hypothèse « l'énergie en kcal se calcule à partir des protéines, glucides, lipides et fibres ».

In [ ]:
nutriments = ["proteines", "glucides", "lipides", "fibres"]
coef, residus = ajuster(produits, "energie_kcal", nutriments)
print(coef.round(3))

fig, ax = plt.subplots(figsize=(10, 3.6))
ax.bar(range(len(produits)), residus, color=LIOUAI["vert"])
ax.set_xticks(range(len(produits)), produits["produit"], rotation=90, fontsize=7)
ax.set_ylabel("résidu (kcal)"); ax.set_title("Écart entre l'étiquette et la relation trouvée, produit par produit")
plt.tight_layout(); plt.show()

Une seule fiche très fausse suffit à tordre toute la relation : les coefficients n'ont pas de sens, et **tous** les produits ont des résidus importants. On procède donc pas à pas : on retire le produit le plus suspect, on recalcule, et on recommence tant qu'un résidu dépasse ce que les arrondis peuvent expliquer.

**Quelle tolérance ?** L'énergie est arrondie à 0,5 kcal près, chaque nutriment à 0,05 g près. Avec des coefficients de l'ordre de 4 à 9, les arrondis peuvent produire un écart de 1 à 2 kcal. On prend un seuil de 3 kcal.

In [ ]:
def ajuster_robuste(tableau, cible, explicatives, seuil):
    # Retire un à un les produits au plus grand résidu, tant qu'un résidu dépasse le seuil
    retenus = tableau.copy()
    suspects = []
    while True:
        coef, residus = ajuster(retenus, cible, explicatives)
        pire = np.argmax(np.abs(residus))
        if abs(residus[pire]) <= seuil:
            return coef, suspects
        suspects.append((retenus["produit"].iloc[pire], round(float(residus[pire]), 2)))
        retenus = retenus.drop(retenus.index[pire])

coef_kcal, suspects_kcal = ajuster_robuste(produits, "energie_kcal", nutriments, seuil=3)
print("Coefficients :", coef_kcal.round(3).to_dict())
print("Produits suspects :", suspects_kcal)

**Activité** : Reconnaissez-vous les coefficients ? Que s'est-il sans doute passé pour le produit suspect ? Regardez sa ligne dans le tableau.

In [ ]:
produits[produits["produit"].isin([s for s, _ in suspects_kcal])][["produit", "energie_kj", "energie_kcal"] + nutriments]

### Les autres relations

In [ ]:
coef_kj, suspects_kj = ajuster_robuste(produits, "energie_kj", nutriments, seuil=3)
print("kJ  :", coef_kj.round(3).to_dict(), " suspects :", suspects_kj)

coef_sel, suspects_sel = ajuster_robuste(produits, "sel", ["sodium"], seuil=0.02)
print("sel :", coef_sel.round(3).to_dict(), " suspects :", suspects_sel)

print("\nRapport kJ / kcal, produit par produit (sans les fiches suspectes) :")
propres = produits[~produits["produit"].isin([s for s, _ in suspects_kcal + suspects_sel])]
rapport = propres["energie_kj"] / propres["energie_kcal"]
print(f"  de {rapport.min():.3f} à {rapport.max():.3f}  (1 kcal = 4,184 kJ en physique)")

### Après nettoyage : le rang

In [ ]:
sv_propre = valeurs_singulieres(propres[colonnes])
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.semilogy(range(1, 11), sv, "o-", label="toutes les fiches")
ax.semilogy(range(1, 11), sv_propre, "s-", label="sans les fiches suspectes")
ax.set_xlabel("rang de la valeur singulière"); ax.set_ylabel("valeur (échelle log)"); ax.legend(); ax.grid(alpha=0.3)
ax.set_title("Les erreurs de saisie masquaient des relations")
plt.tight_layout(); plt.show()
print("Valeurs singulières après nettoyage :", sv_propre.round(4))

### Un piège : trop peu de produits

Supposons qu'on n'ait reçu que 8 fiches, toutes correctes. Avec 10 colonnes et 8 lignes, que dit la méthode ?

In [ ]:
petit = propres.sample(8, random_state=3)
print("Valeurs singulières avec 8 produits :", valeurs_singulieres(petit[colonnes]).round(4))

coef_faux, r_faux = ajuster(petit, "sucres", ["lipides", "glucides", "fibres", "proteines", "sodium", "acides_gras_satures"])
print("\n« Relation » trouvée pour les sucres :", coef_faux.round(2).to_dict())
print("Résidu maximal sur ces 8 produits :", np.abs(r_faux).max().round(3), "g")
autres = propres.drop(petit.index)
B_autres = np.column_stack([np.ones(len(autres)), autres[coef_faux.index[1:]].to_numpy(dtype=float)])
print("Erreur moyenne de cette « relation » sur les autres produits :",
      np.abs(autres["sucres"].to_numpy() - B_autres @ coef_faux.to_numpy()).mean().round(1), "g")

**Activité** : Avec 8 produits, chaque colonne est un vecteur de $\mathbb{R}^8$, et une fois centrée elle vit dans un espace de dimension 7. Combien de colonnes indépendantes peut-il y avoir, au plus ? Combien de « relations » obtient-on alors forcément, quelles que soient les données ? Pourquoi la relation trouvée pour les sucres est-elle absurde sur les autres produits ?

Indiquez vos réponses et justifiez-les.

## ⑤ Interpréter

In [ ]:
print(f"Votre pari : {pari_nombre_relations} relation(s), {pari_fiches_erronees} fiche(s) erronée(s)")
print(f"Résultat   : 3 relations (kcal, kJ, sel), {len(suspects_kcal) + len(suspects_sel)} fiches erronées")

**Activité** :

1. Les coefficients de la relation pour les kcal sont les « facteurs de conversion » imposés par la réglementation européenne sur l'étiquetage. Le résultat de l'algèbre linéaire est-il cohérent avec ce que vous savez (les lipides sont « plus énergétiques ») ?
2. Pourquoi le rapport kJ / kcal n'est-il pas exactement 4,184 ? Une colonne peut-elle être redondante **sans** être proportionnelle à une autre ?
3. Pourquoi les erreurs de saisie « masquaient-elles » des relations sur le graphique des valeurs singulières ?
4. Quel est le danger de chercher des relations avec trop peu de produits ? Quel nombre de produits minimal conseilleriez-vous ?

Indiquez vos réponses et justifiez-les.

### Note à l'équipe données

Rédigez une note d'une demi-page, sans jargon mathématique, qui comprend :

- les colonnes que l'application peut recalculer, et les formules ;
- les fiches à faire corriger par les fabricants, et l'erreur probable pour chacune ;
- un contrôle automatique à mettre en place à chaque nouvelle fiche, avec ses seuils de tolérance ;
- les limites de l'analyse (taille de l'échantillon, arrondis, relations non linéaires).

Indiquez vos réponses et justifiez-les.

## ⑥ Critiquer et généraliser

- **Ce que la méthode ne voit pas.** Les sucres sont toujours inférieurs aux glucides, et les acides gras saturés aux lipides. Ce sont des **inégalités**, pas des relations linéaires : le rang ne les détecte pas. Proposez un contrôle pour elles.
- **Le seuil.** Que se passe-t-il pour l'ajustement robuste avec un seuil de 0,5 kcal ? de 50 kcal ? Comment justifier un seuil auprès de l'équipe ?
- **Généraliser.** Écrivez une fonction `controler_fiche(fiche)` qui reçoit un nouveau produit (un dictionnaire) et renvoie la liste des anomalies détectées par les trois relations et les deux inégalités. Testez-la sur une fiche correcte et sur une fiche où le sel a été saisi en milligrammes.

In [ ]:
def controler_fiche(fiche):
    # À compléter : renvoie une liste de messages d'anomalie (vide si la fiche est cohérente)
    ...

## Corrigé

*À n'ouvrir qu'après avoir cherché.*

In [ ]:
def controler_fiche(fiche):
    anomalies = []
    for nom, coef, seuil in [("energie_kcal", coef_kcal, 3), ("energie_kj", coef_kj, 3)]:
        attendu = coef["constante"] + sum(coef[n] * fiche[n] for n in nutriments)
        if abs(fiche[nom] - attendu) > seuil:
            anomalies.append(f"{nom} : {fiche[nom]} sur l'étiquette, environ {attendu:.0f} attendu")
    attendu_sel = coef_sel["constante"] + coef_sel["sodium"] * fiche["sodium"]
    if abs(fiche["sel"] - attendu_sel) > 0.02:
        anomalies.append(f"sel : {fiche['sel']} g sur l'étiquette, environ {attendu_sel:.2f} g attendu (2,5 × sodium)")
    if fiche["sucres"] > fiche["glucides"]:
        anomalies.append("sucres supérieurs aux glucides")
    if fiche["acides_gras_satures"] > fiche["lipides"]:
        anomalies.append("acides gras saturés supérieurs aux lipides")
    return anomalies

correcte = propres.iloc[0][colonnes].to_dict()
print("Fiche correcte   :", controler_fiche(correcte) or "aucune anomalie")
erronee = dict(correcte, sel=correcte["sel"] * 1000)
print("Sel en mg        :", controler_fiche(erronee))

for seuil in [0.5, 50]:
    _, s = ajuster_robuste(produits, "energie_kcal", nutriments, seuil)
    print(f"Seuil {seuil} kcal : produits écartés {[nom for nom, _ in s]}")
print("Trop petit, le seuil écarte des fiches correctes à cause des arrondis ; trop grand, il laisse passer des erreurs.")

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Repérer les sophismes et mauvais usages du calcul": 1,
    "Identifier erreurs systématiques et aléatoires": 1,
    "Expliquer par écrit": 1,
    "Identifier, énoncer et justifier les hypothèses": 1,
    "Combiner des outils pour produire le résultat": 1,
    "Énoncer le résultat dans le contexte du problème": 1,
    "Corroborer ses résultats": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "M5-projet-redondance", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")